# Plotting PyRK Data

First, you'll need to import PyTables, the hdf5 package.

In [5]:
import tables as tb
!pip install bokeh

     |████████████████████████████████| 8.3MB 16.3MB/s eta 0:00:01
     |████████████████████████████████| 12.4MB 25.7MB/s eta 0:00:01
     |████████████████████████████████| 92kB 10.2MB/s eta 0:00:01
     |████████████████████████████████| 348kB 27.1MB/s eta 0:00:01
You should consider upgrading via the 'pip install --upgrade pip' command.


Next, you'll need a plotting tool. This example uses Bokeh.

In [6]:
from bokeh.plotting import figure, output_notebook, show
output_notebook()

Loading BokehJS ...

Create a variable that will act as a 'handle' to the database. You'll 
need to define the path to the file holding the output database you 
are interested in. It must be an hdf5 file created by pyrk.

In [7]:
filepath='./other.h5'
db=tb.open_file(filepath)

FileNotFoundError: ``/home/jmcfaul/pyrk/examples/sfr/other.h5`` does not exist

To see a list of all the Groups and Tables in the database, 
try the following command. It will list the nodes held within 
the root ('/') node.

In [ ]:
db.list_nodes('/')

Create variables to act as handles to the tables you are interested in. 

In [ ]:
sim_info_table = db.root.metadata.sim_info
power_table = db.root.metadata.sim_timeseries
ne_table = db.root.neutronics.neutronics_params
th_table = db.root.th.th_timeseries

Interesting information about the simulation can be found from the sim_info table:

In [ ]:
print(sim_info_table.col('t_feedback'))
print(sim_info_table.col('simhash'))
# print(sim_info_table.col('inputblob')) # note that this one contains the full input file that you ran....
print(sim_info_table.col('timestamp')) # this is unix epoch time. 
print(sim_info_table.col('humantime')) # human time is easier to read
print(sim_info_table.col('plotdir'))
print(sim_info_table.col('t0'))
print(sim_info_table.col('tf'))
print(sim_info_table.col('dt'))
print(sim_info_table.col('n_pg'))
print(sim_info_table.col('n_dg'))
print(sim_info_table.col('iso'))
print(sim_info_table.col('e'))

You can also list the names of the columns available in a table:

In [ ]:
th_table.coldescrs.keys()
temp_inlet = [v['temp'] for v in th_table.iterrows() if v['component'] == b'inlet'] 
# if you have opened the notebook with python 2, I think you can drop the b before the string
# if you have the notebook open in python 3, then you need the b.

Create an array of your x index. For example, let's plot everything against time.

In [ ]:
x = db.root.th.th_timeseries.col('t_idx')


In [ ]:
temp_inlet = [v['temp'] for v in th_table.iterrows() if v['component'] == b'inlet']
temp_fuel = [v['temp'] for v in th_table.iterrows() if v['component'] == b'fuel']
temp_cool = [v['temp'] for v in th_table.iterrows() if v['component'] == b'cool']

#output_file("bokeh_example.html", title="bokeh example")

TOOLS = "pan,wheel_zoom,box_zoom,reset,save,box_select"

p = figure(title="Bokeh Example", tools=TOOLS)

p.circle(x, temp_inlet, legend="inlet")
p.line(x, temp_inlet, legend="inlet")

p.line(x, temp_fuel, legend="fuel",
    line_dash=[4, 4], line_color="orange", line_width=2)
p.square(x, temp_cool, legend="cool", fill_color=None, line_color="green")
p.line(x, temp_cool, legend="cool", line_color="green")

show(p)  # open a browser

In [ ]:
rho_tot = [v['rho_tot'] for v in ne_table.iterrows()]
rho_ext = [v['rho_ext'] for v in ne_table.iterrows()]

#output_file("bokeh_example.html", title="bokeh example")

TOOLS = "pan,wheel_zoom,box_zoom,reset,save,box_select"

p = figure(title="Reactivity", tools=TOOLS)

p.line(x, rho_tot, legend="total reactivity", line_color="red")
p.line(x, rho_ext, legend="external reactivity", line_color="blue")

show(p)  # open a browser

In [ ]:
power = [v['power'] for v in power_table.iterrows()]

#output_file("bokeh_example.html", title="bokeh example")

TOOLS = "pan,wheel_zoom,box_zoom,reset,save,box_select"

p = figure(title="Reactivity and Power", tools=TOOLS)

p.line(x, rho_tot, legend="total reactivity", line_color="red")
p.line(x, rho_ext, legend="external reactivity", line_color="blue")
p.line(x, power, legend="power", line_color="blue")

show(p)  # open a browser